# SEC MAIS — Sprint 3: Modelagem Analítica e Disponibilização
## Camada de Consumo (Serving Layer) no Data Warehouse PostgreSQL (Docker) & Marts

> **Projeto de TCC / Secretaria de Educação do Estado da Bahia (SEC-BA)**  
> Este notebook consolida a **Sprint 3**, conectando a base consolidada da **Camada Ouro** (`fct_estudantes_beneficios_integrados`) ao banco de dados relacional **PostgreSQL (em container Docker)** e disponibilizando as **visões analíticas de negócio (marts)** para consumo por ferramentas de BI (Power BI, Metabase), relatórios e gestores educacionais.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine, text

# Localiza os caminhos das pastas base do projeto
def localizar_pastas():
    candidatos_base = [
        Path.cwd() / "sec-mais",
        Path.cwd().parent / "sec-mais",
        Path.cwd(),
        Path.cwd().parent,
    ]
    for c in candidatos_base:
        if (c / "armazenamento").exists():
            return c.resolve(), (c / "armazenamento").resolve()
    base = Path.cwd().resolve()
    return base, base / "armazenamento"

DIR_PROJETO, DIR_STORAGE = localizar_pastas()
DIR_OURO = DIR_STORAGE / "ouro"

# Credenciais de conexão com o PostgreSQL DW local (Docker)
PG_HOST = os.getenv("PG_DW_HOST", "localhost")
PG_PORT = int(os.getenv("PG_DW_PORT", 5432))
PG_DB = os.getenv("PG_DW_DATABASE", "sec_mais_dw")
PG_USER = os.getenv("PG_DW_USER", "postgres")
PG_PASS = os.getenv("PG_DW_PASSWORD", "secmais_password")

DATABASE_URI = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
engine = create_engine(DATABASE_URI)

print(f"📁 Raiz do Projeto: {DIR_PROJETO}")
print(f"🔌 PostgreSQL DW:  {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# Teste de conectividade com o banco
try:
    with engine.connect() as conn:
        res = conn.execute(text("SELECT version();")).scalar()
        print(f"✅ Conexão estabelecida com sucesso com o PostgreSQL Docker!")
        print(f"   Versão: {res}")
except Exception as e:
    print(f"❌ Erro ao conectar ao container PostgreSQL: {e}")
    print("   Verifique se o container está rodando via 'docker compose up -d'.")

---  
## 📌 1. Leitura e Validação da Camada Ouro (Arquivo Parquet Integrado)
Carregamento da tabela fato analítica `fct_estudantes_beneficios_integrados.parquet` gerada na Sprint 2.

In [ ]:
caminho_ouro = DIR_OURO / "fct_estudantes_beneficios_integrados.parquet"

if not caminho_ouro.exists():
    raise FileNotFoundError(f"Arquivo da Camada Ouro não encontrado: {caminho_ouro}. Execute a pipeline da Sprint 2 primeiro.")

df_ouro = pd.read_parquet(caminho_ouro)
print(f"✅ Base Ouro lida com sucesso: {len(df_ouro):,} registros e {len(df_ouro.columns)} colunas.")

# Padroniza os nomes das colunas para minúsculo para convenção PostgreSQL
df_ouro.columns = [c.lower().strip() for c in df_ouro.columns]

display(df_ouro.head(5))

---  
## 📌 2. Ingestão dos Dados no PostgreSQL (Schema `ouro`)
Persistência física na tabela relacional `ouro.fct_estudantes_beneficios_integrados` e indexação de chaves para consultas de alta performance.

In [ ]:
with engine.begin() as conn:
    # Garante a existência dos schemas de trabalho
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS ouro;"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS marts;"))
    
    # Grava o DataFrame na tabela ouro.fct_estudantes_beneficios_integrados
    print("⏳ Inserindo registros na tabela ouro.fct_estudantes_beneficios_integrados...")
    df_ouro.to_sql(
        name="fct_estudantes_beneficios_integrados",
        con=conn,
        schema="ouro",
        if_exists="replace",
        index=False,
        chunksize=5000
    )
    
    # Cria índices estratégicos nas chaves de junção e campos analíticos
    print("⚡ Criando índices de busca rápida...")
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_fct_matricula ON ouro.fct_estudantes_beneficios_integrados (matricula);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_fct_cpf ON ouro.fct_estudantes_beneficios_integrados (cpf_estudante);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_fct_total_prog ON ouro.fct_estudantes_beneficios_integrados (total_programas_recebidos);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_fct_municipio ON ouro.fct_estudantes_beneficios_integrados (municipio);"))

print(f"✅ Tabela ouro.fct_estudantes_beneficios_integrados criada e indexada com sucesso ({len(df_ouro):,} linhas)!")

---  
## 📌 3. Criação das Visões Analíticas (Data Marts no Schema `marts`)
Implementação das regras de negócio e visões materializadas para os gestores da SEC-BA:
1. **`marts.vw_resumo_sobreposicao`**: Distribuição de estudantes por volume de benefícios (1, 2 ou 3).
2. **`marts.vw_estudantes_multiplos_beneficios`**: Lista nominal e detalhada de estudantes em múltiplos programas.
3. **`marts.vw_distribuicao_geografica_beneficios`**: Métricas financeiras e de cobertura por município baiano.
4. **`marts.vw_transicao_pe_de_meia_upt`**: Cruzamento entre beneficiários do Pé-de-Meia e matriculados no UPT.

In [ ]:
ddl_views = """
-- 1. Visão de Resumo e Indicadores de Sobreposição de Benefícios
CREATE OR REPLACE VIEW marts.vw_resumo_sobreposicao AS
SELECT
    total_programas_recebidos AS qtd_programas_acumulados,
    COUNT(*) AS total_estudantes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentual_base,
    SUM(CASE WHEN beneficio_pe_de_meia THEN 1 ELSE 0 END) AS total_com_pe_de_meia,
    SUM(CASE WHEN beneficio_upt THEN 1 ELSE 0 END) AS total_com_upt,
    SUM(CASE WHEN beneficio_bolsa_presenca THEN 1 ELSE 0 END) AS total_com_bolsa_presenca
FROM ouro.fct_estudantes_beneficios_integrados
GROUP BY total_programas_recebidos
ORDER BY total_programas_recebidos;

-- 2. Visão Detalhada de Estudantes em Múltiplos Benefícios
CREATE OR REPLACE VIEW marts.vw_estudantes_multiplos_beneficios AS
SELECT
    matricula,
    nome_estudante,
    cpf_estudante,
    municipio,
    total_programas_recebidos,
    beneficio_pe_de_meia,
    status_pe_de_meia,
    beneficio_bolsa_presenca,
    status_bolsa_presenca,
    valor AS valor_bolsa_presenca,
    beneficio_upt,
    numero_inscricao AS inscricao_upt,
    status_upt
FROM ouro.fct_estudantes_beneficios_integrados
WHERE total_programas_recebidos >= 2
ORDER BY total_programas_recebidos DESC, nome_estudante;

-- 3. Visão Geográfica: Cobertura por Município
CREATE OR REPLACE VIEW marts.vw_distribuicao_geografica_beneficios AS
SELECT
    COALESCE(municipio, 'NÃO INFORMADO') AS municipio,
    COUNT(*) AS total_atendimentos,
    SUM(CASE WHEN beneficio_bolsa_presenca THEN 1 ELSE 0 END) AS atendidos_bolsa_presenca,
    SUM(CASE WHEN beneficio_pe_de_meia THEN 1 ELSE 0 END) AS atendidos_pe_de_meia,
    SUM(CASE WHEN beneficio_upt THEN 1 ELSE 0 END) AS atendidos_upt,
    SUM(CASE WHEN total_programas_recebidos >= 2 THEN 1 ELSE 0 END) AS atendidos_multiplos_beneficios,
    ROUND(AVG(COALESCE(valor, 0)), 2) AS ticket_medio_bolsa_presenca,
    SUM(COALESCE(valor, 0)) AS valor_total_bolsa_presenca
FROM ouro.fct_estudantes_beneficios_integrados
GROUP BY COALESCE(municipio, 'NÃO INFORMADO')
ORDER BY total_atendimentos DESC;

-- 4. Visão de Transição Educacional: Ensino Médio (Pé-de-Meia) -> Superior (UPT)
CREATE OR REPLACE VIEW marts.vw_transicao_pe_de_meia_upt AS
SELECT
    matricula,
    nome_estudante,
    cpf_estudante,
    status_pe_de_meia,
    numero_inscricao AS inscricao_upt,
    status_upt,
    beneficio_bolsa_presenca
FROM ouro.fct_estudantes_beneficios_integrados
WHERE beneficio_pe_de_meia = TRUE
  AND beneficio_upt = TRUE
ORDER BY nome_estudante;
"""

with engine.begin() as conn:
    conn.execute(text(ddl_views))

print("✅ Visões analíticas criadas com sucesso no schema 'marts'!")

---  
## 📌 4. Validação e Consultas Analíticas Diretamente no Data Warehouse (Pronto para o Relatório do TCC)

Executamos consultas SQL reais contra as views do PostgreSQL para demonstrar a disponibilização analítica dos dados.

In [ ]:
# Consulta 1: Indicador de Sobreposição de Benefícios
with engine.connect() as conn:
    df_view1 = pd.read_sql("SELECT * FROM marts.vw_resumo_sobreposicao;", con=conn)

print("===================================================================================")
print("      INDICADOR 1: DISTRIBUIÇÃO E SOBREPOSIÇÃO DE BENEFÍCIOS EDUCACIONAIS          ")
print("===================================================================================")
display(df_view1)

In [ ]:
# Consulta 2: Top 10 Municípios Baianos com Maior Volume de Concessões
with engine.connect() as conn:
    df_view2 = pd.read_sql("SELECT * FROM marts.vw_distribuicao_geografica_beneficios LIMIT 10;", con=conn)

print("===================================================================================")
print("      INDICADOR 2: TOP 10 MUNICÍPIOS COM MAIOR CONCENTRAÇÃO DE BENEFÍCIOS          ")
print("===================================================================================")
display(df_view2)

In [ ]:
# Consulta 3: Transição Pé-de-Meia -> UPT (Ensino Médio para Superior)
with engine.connect() as conn:
    df_view3 = pd.read_sql("SELECT * FROM marts.vw_transicao_pe_de_meia_upt LIMIT 10;", con=conn)

print("===================================================================================")
print("      INDICADOR 3: AMOSTRA DE ESTUDANTES NA TRANSIÇÃO PÉ-DE-MEIA -> UPT            ")
print("===================================================================================")
display(df_view3)